In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mode de test local de Qiskit Runtime

<span id="test-locally"></span>


<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
Utilise le mode de test local (disponible avec `qiskit-ibm-runtime` v0.22.0 ou ultérieur) pour tester tes programmes avant de les peaufiner et de les envoyer vers du matériel quantique réel. Après avoir utilisé le mode de test local pour vérifier ton programme, il te suffit de changer le nom du backend pour l'exécuter sur un QPU.

Pour utiliser le mode de test local, spécifie l'un des fake backends de ``qiskit_ibm_runtime.fake_provider`` ou un backend Qiskit Aer lors de l'instanciation d'une primitive Qiskit Runtime ou d'une session.

- **Fake backends** : Les [fake backends](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) dans ``qiskit_ibm_runtime.fake_provider`` imitent le comportement des QPUs IBM&reg; en utilisant des instantanés de QPU. Ces instantanés contiennent des informations importantes sur le QPU, telles que la carte de couplage, les portes de base et les propriétés des qubits, utiles pour tester le transpiler et effectuer des simulations bruitées du QPU. Le modèle de bruit issu de l'instantané est automatiquement appliqué lors de la simulation.
- **Simulateur Aer** : Les simulateurs de [Qiskit Aer](/guides/simulate-with-qiskit-aer) offrent des simulations plus performantes, capables de gérer des circuits plus grands et des [modèles de bruit personnalisés](/guides/build-noise-models). Une liste des méthodes de simulation disponibles est accessible lorsque tu utilises `AerSimulator` en mode de test local. Consulte l'[exemple du mode de simulation Clifford](#clifford-sim), qui montre comment simuler efficacement des circuits Clifford avec un grand nombre de qubits.

    <details>
    <summary>**Liste des méthodes de simulation disponibles dans Qiskit Aer**</summary>

    Consulte la documentation d'[`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) pour plus d'informations.

    * ``"automatic"`` : Méthode de simulation par défaut. Sélectionne automatiquement la méthode de simulation en fonction du circuit et du modèle de bruit.

    * ``"statevector"`` : Une simulation par vecteur d'état dense qui peut échantillonner les résultats de mesure depuis des circuits *idéaux* avec toutes les mesures à la fin du circuit. Pour les simulations bruitées, chaque shot échantillonne aléatoirement un circuit bruité à partir du modèle de bruit.

    * ``"density_matrix"`` : Une simulation par matrice densité qui peut échantillonner les résultats de mesure depuis des circuits *bruités* avec toutes les mesures à la fin du circuit.

    * ``"stabilizer"`` : Un simulateur d'état stabilisateur de Clifford efficace, capable de simuler des circuits Clifford bruités si toutes les erreurs du modèle de bruit sont également des erreurs de Clifford.

    * ``"extended_stabilizer"`` : Un simulateur approché pour les circuits Clifford + T, basé sur la décomposition de l'état en un état stabilisateur de rang. Le nombre de termes croît avec le nombre de portes non-Clifford (T).

    * ``"matrix_product_state"`` : Un simulateur de vecteur d'état par réseau de tenseurs qui utilise une représentation en Matrix Product State (MPS) pour l'état. Cela peut être fait avec ou sans troncature des dimensions de liaison MPS, selon les options du simulateur. Par défaut, il n'y a pas de troncature.

    * ``"unitary"`` : Une simulation dense de matrice unitaire d'un circuit idéal. Cette méthode simule la matrice unitaire du circuit lui-même, plutôt que l'évolution d'un état quantique initial. Elle ne peut simuler que des portes ; elle ne prend pas en charge les mesures, les réinitialisations ou le bruit.

    * ``"superop"`` : Une simulation dense de matrice de superopérateur d'un circuit idéal ou bruité. Cette méthode simule la matrice de superopérateur du circuit lui-même, plutôt que l'évolution d'un état quantique initial. Elle peut simuler des portes et des réinitialisations idéales et bruitées, mais ne prend pas en charge les mesures.

    * ``"tensor_network"`` : Une simulation par réseau de tenseurs qui prend en charge à la fois le vecteur d'état et la matrice densité. Actuellement disponible uniquement sur GPU et accélérée par les APIs cuQuantum `cuTensorNet`.
    </details>

> **Note:** - Tu peux spécifier toutes les options de Qiskit Runtime en mode de test local. Cependant, toutes les options sauf `shots` sont ignorées lors de l'exécution sur un simulateur local.
> - Il est recommandé d'installer Qiskit Aer avant d'utiliser des fake backends ou des simulateurs Aer en exécutant `pip install qiskit-aer`. Les fake backends utilisent les simulateurs Aer en arrière-plan si disponibles, afin de tirer parti de leurs performances.


## Exemple avec les fake backends

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## Exemples avec AerSimulator {#aerSimulator-examples}

Exemple avec sessions, sans bruit :

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

Pour simuler avec du bruit, spécifie un QPU (matériel quantique) et soumets-le à Aer. Aer construit un modèle de bruit basé sur les données de calibration de ce QPU et instancie un backend Aer avec ce modèle. Si tu préfères, tu peux [construire un modèle de bruit](/guides/build-noise-models).

> **Caution:** Un QPU peut être affecté par différents types de bruit. Le modèle de bruit de Qiskit Aer utilisé ici n'en simule qu'une partie et sera donc probablement moins sévère que le bruit d'un vrai QPU.
> 
> Pour plus de détails sur les erreurs incluses lors de l'initialisation d'un modèle de bruit à partir d'un QPU, consulte la référence API Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend).

Exemple avec bruit :

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### Simulation Clifford
Comme les circuits Clifford peuvent être simulés efficacement avec des résultats vérifiables, la simulation Clifford est un outil très utile. Pour un exemple approfondi, consulte [Simulation efficace de circuits stabilisateurs avec les primitives Qiskit Aer](/guides/simulate-stabilizer-circuits).

Exemple :

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## Étapes suivantes
> **Tip:** - Consulte des [exemples détaillés de primitives.](/guides/primitives-examples)
>     - Lis [Migrer vers les primitives V2](/guides/v2-primitives).
>     - Pratique avec les primitives en suivant la [leçon sur les fonctions de coût](/learning/courses/variational-algorithm-design/cost-functions) dans IBM Quantum Learning.
>     - Apprends à transpiler localement dans la section [Transpiler](/guides/transpile).
>     - Essaie le tutoriel [Comparer les paramètres du transpilateur](/guides/circuit-transpilation-settings#compare-transpiler-settings).